In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error


#### we want to extract 8 columns and train the model from those featuers 
#### batting_team
#### bowling team
#### city
#### current score
#### ball left
#### wickets left
#### current rr
#### last five

In [22]:
df = pd.read_csv('data.csv')
df


,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,venue
0,2,Australia,Sri Lanka,0.1,0,0,NaN,Melbourne Cricket Ground
1,2,Australia,Sri Lanka,0.2,0,0,NaN,Melbourne Cricket Ground
2,2,Australia,Sri Lanka,0.3,1,0,NaN,Melbourne Cricket Ground
3,2,Australia,Sri Lanka,0.4,2,0,NaN,Melbourne Cricket Ground
4,2,Australia,Sri Lanka,0.5,0,0,NaN,Melbourne Cricket Ground
...,...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,0,Colombo,R Premadasa Stadium
110945,3222,Sri Lanka,Australia,19.4,0,0,Colombo,R Premadasa Stadium
110946,3222,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,R Premadasa Stadium
110947,3222,Sri Lanka,Australia,19.6,2,0,Colombo,R Premadasa Stadium


#### Handling missing values

In [23]:
df.isnull().sum()


match_id               0
batting_team           0
bowling_team           0
ball                   0
runs                   0
player_dismissed       0
city                8549
venue                  0
dtype: int64

In [24]:
df[df['city'].isnull()]['venue'].value_counts()

venue
Dubai International Cricket Stadium        3092
Pallekele International Cricket Stadium    2066
Melbourne Cricket Ground                   1453
Sydney Cricket Ground                       749
Adelaide Oval                               498
Harare Sports Club                          372
Sylhet International Cricket Stadium        128
Sharjah Cricket Stadium                     127
Carrara Oval                                 64
Name: count, dtype: int64

### From this we can see that we can fill city with the first word of venue_name

In [25]:
cities = np.where(df['city'].isnull(),df['venue'].str.split().apply(lambda x:x[0]),df['city'])
df['city'] = cities
df.isnull().sum()

match_id            0
batting_team        0
bowling_team        0
ball                0
runs                0
player_dismissed    0
city                0
venue               0
dtype: int64

#### Now there is no need of venue column. SO dropping it

In [26]:
df.drop(columns=['venue'],inplace=True)
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne
...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,0,Colombo
110945,3222,Sri Lanka,Australia,19.4,0,0,Colombo
110946,3222,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo
110947,3222,Sri Lanka,Australia,19.6,2,0,Colombo


#### we only want to use those cities which have atleast host 5 matches i.e 600 deliveries
#### dropping cities with less than 600 deliveries

In [27]:
eligible_cities = df['city'].value_counts()[df['city'].value_counts() > 600].index.tolist()
df = df[df['city'].isin(eligible_cities)]
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne
...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,0,Colombo
110945,3222,Sri Lanka,Australia,19.4,0,0,Colombo
110946,3222,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo
110947,3222,Sri Lanka,Australia,19.6,2,0,Colombo


#### creating current score column

In [28]:
df['runs'] = pd.to_numeric(df['runs'])
df['current_score'] = df.groupby('match_id')['runs'].cumsum()
df 

C:\Users\HP\AppData\Local\Temp\ipykernel_12316\367082088.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['runs'] = pd.to_numeric(df['runs'])
C:\Users\HP\AppData\Local\Temp\ipykernel_12316\367082088.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['current_score'] = df.groupby('match_id')['runs'].cumsum()


,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3
...,...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,0,Colombo,125
110945,3222,Sri Lanka,Australia,19.4,0,0,Colombo,125
110946,3222,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,125
110947,3222,Sri Lanka,Australia,19.6,2,0,Colombo,127


creating balls left column

In [29]:
#splitting over no and ball no from ball column
df['over'] = df['ball'].apply(lambda x:str(x).split(".")[0])
df['ball_no'] = df['ball'].apply(lambda x:str(x).split(".")[1])

#calculating balls bowled with the help of over and ball no
df['balls_bowled'] = (df['over'].astype('int')*6) + df['ball_no'].astype('int')

#calculating balls left by subtracting balls bowled from total balls in an innings(120)
df['balls_left'] = 120 - df['balls_bowled']

#due to extras(wides and no balls) ball bowled may exceed 120..so ball left will be in negative.so converting to zero

df['balls_left'] = df['balls_left'].apply(lambda x:0 if x<0 else x)
df

C:\Users\HP\AppData\Local\Temp\ipykernel_12316\2509475007.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['over'] = df['ball'].apply(lambda x:str(x).split(".")[0])
C:\Users\HP\AppData\Local\Temp\ipykernel_12316\2509475007.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ball_no'] = df['ball'].apply(lambda x:str(x).split(".")[1])
C:\Users\HP\AppData\Local\Temp\ipykernel_12316\2509475007.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115
...,...,...,...,...,...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,0,Colombo,125,19,3,117,3
110945,3222,Sri Lanka,Australia,19.4,0,0,Colombo,125,19,4,118,2
110946,3222,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,125,19,5,119,1
110947,3222,Sri Lanka,Australia,19.6,2,0,Colombo,127,19,6,120,0


creating wickets_left column

In [30]:
df['player_dismissed'] = (
    df['player_dismissed']
    .ne('0')
    .astype(int)
)

df['player_dismissed'] = df.groupby(['match_id', 'batting_team'])['player_dismissed'].cumsum()

df['wickets_left'] = 10 - df['player_dismissed']

C:\Users\HP\AppData\Local\Temp\ipykernel_12316\3208117847.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['player_dismissed'] = (
C:\Users\HP\AppData\Local\Temp\ipykernel_12316\3208117847.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['player_dismissed'] = df.groupby(['match_id', 'batting_team'])['player_dismissed'].cumsum()
C:\Users\HP\AppData\Local\Temp\ipykernel_12316\3208117847.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

In [31]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left,wickets_left
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119,10
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118,10
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117,10
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116,10
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,8,Colombo,125,19,3,117,3,2
110945,3222,Sri Lanka,Australia,19.4,0,8,Colombo,125,19,4,118,2,2
110946,3222,Sri Lanka,Australia,19.5,0,9,Colombo,125,19,5,119,1,1
110947,3222,Sri Lanka,Australia,19.6,2,9,Colombo,127,19,6,120,0,1


creating current run rate(crr) column

In [32]:
df['crr'] = (df['current_score']*6)/df['balls_bowled']
df

C:\Users\HP\AppData\Local\Temp\ipykernel_12316\1226876470.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['crr'] = (df['current_score']*6)/df['balls_bowled']


,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left,wickets_left,crr
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119,10,0.000000
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118,10,0.000000
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117,10,2.000000
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116,10,4.500000
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115,10,3.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,8,Colombo,125,19,3,117,3,2,6.410256
110945,3222,Sri Lanka,Australia,19.4,0,8,Colombo,125,19,4,118,2,2,6.355932
110946,3222,Sri Lanka,Australia,19.5,0,9,Colombo,125,19,5,119,1,1,6.302521
110947,3222,Sri Lanka,Australia,19.6,2,9,Colombo,127,19,6,120,0,1,6.350000


creating runs in last 5 overs column

In [39]:
df = df.sort_values(['match_id', 'ball'])

df['runs'] = pd.to_numeric(df['runs'], errors='coerce')

df['last_five'] = (
    df.groupby('match_id')['runs']   # 👈 ONLY runs
    .rolling(window=30, min_periods=30)
    .sum()
    .reset_index(level=0, drop=True)
)

In [40]:

df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left,wickets_left,crr,last_five
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119,10,0.000000,NaN
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118,10,0.000000,NaN
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117,10,2.000000,NaN
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116,10,4.500000,NaN
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115,10,3.600000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110944,3222,Sri Lanka,Australia,19.3,1,8,Colombo,125,19,3,117,3,2,6.410256,32.0
110945,3222,Sri Lanka,Australia,19.4,0,8,Colombo,125,19,4,118,2,2,6.355932,32.0
110946,3222,Sri Lanka,Australia,19.5,0,9,Colombo,125,19,5,119,1,1,6.302521,32.0
110947,3222,Sri Lanka,Australia,19.6,2,9,Colombo,127,19,6,120,0,1,6.350000,33.0


creating new column with final score of the match and merging with new final_df 

In [41]:
final_df = df.groupby('match_id').sum()['runs'].reset_index().merge(df,on='match_id')


### removing unwanted columns

In [42]:
final_df=final_df[['batting_team','bowling_team','city','current_score','balls_left','wickets_left','crr','last_five','runs_x']]
final_df.dropna(inplace=True)
final_df.isnull().sum()

batting_team     0
bowling_team     0
city             0
current_score    0
balls_left       0
wickets_left     0
crr              0
last_five        0
runs_x           0
dtype: int64

We have created data frame required for model training and testing.
Now creating model.

Shuffling rows of df to remove bias

In [43]:
final_df = final_df.sample(final_df.shape[0])
final_df.sample(2)

,batting_team,bowling_team,city,current_score,balls_left,wickets_left,crr,last_five,runs_x
38162,India,Sri Lanka,Mumbai,64,68,7,7.384615,30.0,162
25918,Bangladesh,England,Abu Dhabi,49,66,7,5.444444,30.0,124


splitting dataset

In [44]:
# here runs_x is the number of runs scored by the batting team in an inning which we have to predict using other columns
X = final_df.drop(columns=['runs_x']) 
y = final_df['runs_x'] 

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)
X_train

,batting_team,bowling_team,city,current_score,balls_left,wickets_left,crr,last_five
64928,South Africa,Bangladesh,Johannesburg,83,65,7,9.054545,35.0
63432,India,Australia,Durban,128,29,7,8.439560,63.0
55696,South Africa,Pakistan,Lahore,122,11,2,6.715596,39.0
27943,India,England,Birmingham,166,2,2,8.440678,52.0
96344,India,Bangladesh,Dhaka,73,51,7,6.347826,38.0
...,...,...,...,...,...,...,...,...
14195,India,New Zealand,Mount Maunganui,50,89,9,9.677419,46.0
1191,India,England,Bangalore,85,52,8,7.500000,31.0
83835,England,Australia,Melbourne,54,81,7,8.307692,44.0
8763,Australia,India,Sydney,52,79,10,7.609756,36.0


### one hot encoding for categorical variables

In [47]:
trf = ColumnTransformer([
    ('trf',OneHotEncoder(sparse_output=False,drop='first'),['batting_team','bowling_team','city'])
]
,remainder='passthrough')       

Model training


In [48]:
## First preprocess data, second scale features, third train with XGBoost regression model
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',StandardScaler()),
    ('step3',XGBRegressor(n_estimators=1000,learning_rate=0.2,max_depth=12,random_state=1))
])

In [49]:
# model training
pipe.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('step1', ...), ('step2', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('trf', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sp

## model evaluation

In [50]:
# Test set predictions
y_pred = pipe.predict(X_test)


# Calculate metrics
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.9858357310295105
1.919534683227539


### creating .pkl file

In [51]:
import joblib
joblib.dump(pipe, "model.pkl")

['model.pkl']